# 4b. PUSH transfer (DSP 2025-1, eunomia RPC)

In **PUSH** mode the Consumer names a sink in its request and the Provider delivers the data to
it: on start the Provider subscribes to its backend through the PUSH connector, and the backend's
events flow through the Provider's dataplane to the sink. The connectors of this notebook use no
authentication (`setup_catalog_policies.ipynb`).

Lifecycle walked here: `REQUESTED` -> `STARTED` -> `SUSPENDED` -> `STARTED` -> `COMPLETED`, plus
the role refusal of a start sent by the Consumer.

**Runs on its own.** Prerequisites: `setup_catalog_policies.ipynb` and `contract_negotiation.ipynb`,
both guarded. The start needs the demo backend on `127.0.0.1:8082` (subscriptions) and a sink on
`127.0.0.1:1112`.

## 1. Prerequisites

Two guards make the notebook runnable on its own and cheap to rerun in the same kernel:
- the catalog notebook runs unless the kernel already holds its variant (`catalog_setup == "no_auth"`);
  it onboards the participants first if needed;
- the negotiation runs unless the kernel already holds an agreement from `contract_negotiation.ipynb` on the loaded
  dataset (`agreement_source == ("rpc", target_id)`).

In [ ]:
if globals().get("catalog_setup") != "no_auth":
    %run ./setup_catalog_policies.ipynb
if globals().get("agreement_source") != ("rpc", target_id):
    %run ./contract_negotiation.ipynb
print(f"Transferring under agreement {agreement_id}")

### How the transfer RPCs work

As in the negotiation, each participant asks its own agent to send the next DSP message, through
`/dsp/current/transfers/rpc/setup-{request,start,suspension,completion,termination}`. Bodies are
eunomia JSON-LD (`@context` the eunomia context, `@type` `RpcTransfer<Kind>Message`) and the
answer is an `RpcTransferResult`: `request`, the peer's `response`, and `transferAgentModel`, the
caller's own process (`state`, `correlation` with both pids, messages). A request must go to the
Provider of the agreement it names.

Who may send what is fixed by the protocol (DSP 9.1): only the Consumer requests, only the
Provider starts, both may suspend, complete or terminate.

In [ ]:
def transfer_step(response, expected_state):
    """Prints an RpcTransferResult and checks the caller's process reached expected_state."""
    answer = show(response)
    process = answer["transferAgentModel"]
    assert process["state"] == expected_state, f"expected {expected_state}, got {process['state']}"
    print(f"\n==> {process['state']}")
    return process

## 2. Transfer request with a sink (Consumer -> Provider)

The Consumer requests the `http+push` distribution and names, in `dataAddress`, the sink where
the Provider must deliver the data, with the credentials the Provider must present there.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferRequestMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+push",
    "dataAddress": {
        "endpointType": "https://w3id.org/idsa/v4.1/HTTP",
        "endpoint": "http://127.0.0.1:1112",
        "endpointProperties": [
            {"name": "authorization", "value": "TOKEN-ABCDEFG"},
            {"name": "authType", "value": "bearer"},
        ],
    },
}
response = requests.post(f"{data_space_consumer}/dsp/current/transfers/rpc/setup-request", headers=json_header, json=payload)
consumer_transfer = transfer_step(response, "REQUESTED")
transfer_consumer_pid = consumer_transfer["correlation"]["consumerPid"]
transfer_provider_pid = consumer_transfer["correlation"]["providerPid"]
print(f"consumerPid: {transfer_consumer_pid}\nproviderPid: {transfer_provider_pid}")

## 3. Start (Provider -> Consumer)

The Provider subscribes to its backend through the PUSH connector and starts pushing to the
Consumer's sink.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-start", headers=json_header, json=payload)
provider_transfer = transfer_step(response, "STARTED")

## 4. Suspend (Provider -> Consumer)

The Provider pauses delivery; the subscription is dropped until the transfer starts again.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferSuspensionMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"],
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-suspension", headers=json_header, json=payload)
provider_transfer = transfer_step(response, "SUSPENDED")

## 5. Resume (Provider -> Consumer)

A suspended transfer resumes with another start message.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-start", headers=json_header, json=payload)
provider_transfer = transfer_step(response, "STARTED")

## 6. Role check: the Consumer cannot start

Only the Provider sends a `TransferStartMessage`. The Consumer's agent refuses the RPC before
anything goes on the wire and answers a `TransferError` naming the broken rule.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferStartMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
}
response = requests.post(f"{data_space_consumer}/dsp/current/transfers/rpc/setup-start", headers=json_header, json=payload)
answer = show(response, ok=False)
assert response.status_code == 400, "the Consumer must not be able to start a transfer"

## 7. Completion (Provider -> Consumer)

When everything is delivered the Provider completes the transfer. `COMPLETED` is terminal.

In [ ]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferCompletionMessage",
    "consumerPid": transfer_consumer_pid,
    "providerPid": transfer_provider_pid,
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-completion", headers=json_header, json=payload)
provider_transfer = transfer_step(response, "COMPLETED")

## 8. Final state on both sides

Each agent's management API (`/api/v1/transfer-agent/transfer-processes/{id}`) shows its own record
of the process, with every message exchanged.

In [ ]:
for name, agent_url, process in [("Consumer", data_space_consumer, consumer_transfer), ("Provider", data_space_provider, provider_transfer)]:
    view = requests.get(f"{agent_url}/api/v1/transfer-agent/transfer-processes/{process['id']}", headers=json_header).json()
    print(f"{name}: {view['state']}")
print()
print(json.dumps(view, indent=2))